# Pandas Avanzado — Versión Extendida
Incluye más ejemplos y casos prácticos.

## 1. Indexado avanzado y MultiIndex

In [ ]:
import pandas as pd, numpy as np

df = pd.DataFrame({
    'Ciudad':['Madrid','Sevilla','Madrid','Valencia'],
    'Ventas':[100,200,150,130],
    'Unidades':[5,7,6,4]
})
df

### Explicación
Creamos un DataFrame base para ejemplificar operaciones avanzadas: servirá para indexado por etiqueta, posición y construcción de MultiIndex.


In [ ]:
df.loc[df['Ciudad']=='Madrid']

### Explicación
Filtro por condición usando `loc`: selecciona todas las filas donde `Ciudad == "Madrid"`. Mantiene estructura y permite encadenar más selecciones.


In [ ]:
df.iloc[1:3]

### Explicación
Rebanado posicional con `iloc[1:3]`: devuelve filas 1 y 2 (límite superior exclusivo). Útil para submuestras rápidas sin depender de etiquetas.


### MultiIndex

In [ ]:
df_mi = df.set_index(['Ciudad','Unidades'])
df_mi

### Explicación
Construimos un MultiIndex con `set_index(['Ciudad','Unidades'])` para permitir consultas jerárquicas y agregaciones más precisas sobre combinaciones de niveles.


In [ ]:
df_mi.xs('Madrid', level='Ciudad')

### Explicación
`xs('Madrid', level='Ciudad')` extrae la sección del MultiIndex para la ciudad dada. `xs` facilita seleccionar por nivel sin tener que especificar todos los niveles superiores.


## 2. Filtros y expresiones avanzadas

In [ ]:
df[df['Ventas'].between(120,180)]

### Explicación
Uso de `between(120,180)` para crear un filtro inclusivo en ambos extremos sin escribir dos comparaciones. Devuelve filas con ventas en el rango.


In [ ]:
df.query('Ciudad in ["Madrid","Sevilla"] and Ventas > 120')

### Explicación
`query` con `in [...]` y condición extra permite escribir filtros legibles sobre listas y operadores lógicos sin exceso de paréntesis.


In [ ]:
df.eval('valor_medio = Ventas / Unidades')

### Explicación
`eval` crea una nueva columna calculada (`valor_medio`) de forma vectorizada y más concisa que asignar con expresión tradicional. Útil para expresiones largas.


## 3. GroupBy avanzado

In [ ]:
df.groupby('Ciudad').agg(
    media_ventas=('Ventas','mean'),
    total_ventas=('Ventas','sum'),
    max_unidades=('Unidades','max')
)

### Explicación
Agregaciones nombradas con sintaxis de tuplas: cada clave del dict se convierte en nueva columna (`media_ventas`, `total_ventas`, etc.), facilitando resultados claros.


In [ ]:
df.groupby('Ciudad').apply(lambda g: g[g['Ventas']==g['Ventas'].max()])

### Explicación
`apply` por grupo para extraer filas cuya `Ventas` es máxima en cada ciudad; devuelve DataFrame combinado preservando índice original.


## 4. Apply / Transform / Pipe con ejemplos complejos

In [ ]:
def escala(x): return (x - x.min())/(x.max()-x.min())
df['Ventas_norm'] = df.groupby('Ciudad')['Ventas'].transform(escala)
df

### Explicación
`transform` con función de normalización por grupo crea columna comparable entre ciudades. Mantiene número de filas ideal para ingeniería de características.


In [ ]:
(df.pipe(lambda d: d.assign(V2=d['Ventas']*2))
      .pipe(lambda d: d[d['V2']>250]))

### Explicación
Uso encadenado de `pipe` para mantener flujo funcional: primero crea columna derivada (`V2`), luego filtra. `pipe` facilita reordenar pasos y pruebas.


## 5. Ventanas móviles y temporales

In [ ]:
s = pd.Series(range(1,21))
s.rolling(5).sum()

### Explicación
Serie de 1..20 y suma móvil de ventana 5: suaviza fluctuaciones y prepara indicadores agregados temporales.


In [ ]:
s.expanding().mean()

### Explicación
`expanding().mean()` calcula media acumulada progresiva útil para analizar convergencia o tendencias generales.


In [ ]:
s.ewm(alpha=0.2).mean()

### Explicación
`ewm(alpha=0.2).mean()` calcula media exponencial ponderando más los valores recientes; útil para series temporales con reacción rápida.


## 6. Fechas y resampling avanzado

In [ ]:
fechas = pd.date_range('2024-01-01', periods=10, freq='D')
df2 = pd.DataFrame({'Fecha':fechas, 'Ventas':np.random.randint(10,100,10)})
df2

### Explicación
Generamos rango de fechas y DataFrame temporal para practicar resampling y agregaciones sobre períodos homogéneos.


In [ ]:
df2.resample('3D', on='Fecha').sum()

### Explicación
`resample('3D', on='Fecha').sum()` agrupa cada bloque de 3 días tomando la suma de ventas; útil para reducir granularidad manteniendo totales.


## 7. Joins avanzados

In [ ]:
dfA = pd.DataFrame({'id':[1,2,3,4], 'valor':[10,20,30,40]})
dfB = pd.DataFrame({'id':[3,4,5], 'cat':['A','B','C']})
dfA.merge(dfB, on='id', how='outer')

### Explicación
Ejemplo de `merge` externo con columnas diferentes para observar cómo se introducen NaN donde faltan coincidencias (`outer`).


In [ ]:
dfA.merge(dfB, on='id', how='inner')

### Explicación
`inner` merge conserva sólo ids presentes en ambos DataFrames; útil para intersecciones de claves.


## 8. Merge con condiciones

In [ ]:
dfL = pd.DataFrame({'x':[1,5,9]})
dfR = pd.DataFrame({'y':[2,4,6,8]})

(dfL.merge(dfR, how='cross')
    .query('x < y'))

### Explicación
`merge` con `how='cross'` genera producto cartesiano y luego se filtra por condición (`x < y`). Útil para explorar combinaciones posibles.


## 9. Intervalos e interval joins

In [ ]:
intervals = pd.IntervalIndex.from_breaks([0,10,20,30])
df_int = pd.DataFrame({'intervalo': intervals, 'categoria':['baja','media','alta']})
df_int

### Explicación
`IntervalIndex.from_breaks` crea intervalos contiguos; se usa para categorizar valores por rangos y luego mapear a etiquetas descriptivas.


In [ ]:
df_vals = pd.DataFrame({'valor':[3,7,12,25]})
pd.merge_asof(df_vals.sort_values('valor'),
              df_int.assign(start=df_int['intervalo'].apply(lambda x: x.left),
                            end=df_int['intervalo'].apply(lambda x: x.right)
                           ).sort_values('start'),
              left_on='valor', right_on='start')

### Explicación
`merge_asof` realiza unión aproximada ordenada (por cercanía hacia atrás). Aquí se prepara DataFrame de intervalos con límites para asociar cada valor al inicio de intervalo más próximo.


## 10. Optimización

In [ ]:
df_opt = df.copy()
df_opt['Ciudad'] = df_opt['Ciudad'].astype('category')
df_opt.dtypes

### Explicación
Conversión a `category` reduce memoria y acelera operaciones basadas en repetición de valores (comparaciones, groupby), útil en columnas con pocos valores distintos.
